## Imports

In [58]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.metrics import average_precision_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

## Model Comparison

This notebook compares tree-based models on the cleaned credit risk dataset: XGBoost, LightGBM, and Random Forest (baseline result carried over from `01_eda_and_cleaning.ipynb`). All models are trained on the same `X_train`/`X_test` split (`random_state=42`, `stratify=y`) to ensure a fair comparison.

In [59]:
credit_risk_cleaned = pd.read_csv("../data/processed/credit_risk_cleaned.csv")
X = credit_risk_cleaned.drop(columns=['has_late_data_anomaly', 'delinquent_2yrs'])
y = credit_risk_cleaned['delinquent_2yrs']

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### XGBoost — Baseline

Class imbalance is handled via `scale_pos_weight` rather than resampling, consistent with the approach used for Random Forest in the previous notebook.

In [61]:
xgb = XGBClassifier(
    learning_rate=0.1,
    n_estimators=300,
    max_depth=5,
    scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
    random_state=42,
    eval_metric='aucpr'
)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.82      0.89     27876
           1       0.22      0.74      0.34      2002

    accuracy                           0.81     29878
   macro avg       0.60      0.78      0.62     29878
weighted avg       0.93      0.81      0.85     29878



At the default 0.5 threshold, recall for the minority class (0.74) is much higher than for Random Forest's default-threshold results — but precision is low (0.22), so threshold tuning is needed before drawing conclusions.

### Threshold Tuning

Same approach as in the previous project: scan thresholds from 0.05 to 0.95 and pick the one that maximizes F1 on the test set.

In [62]:
y_proba = xgb.predict_proba(X_test)[:, 1]

best_f1 = 0
best_threshold = 0.5

for threshold in np.arange(0.05, 0.95, 0.05):
    y_pred_t = (y_proba >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Best threshold: {best_threshold:.2f}, F1: {best_f1:.3f}")
print(f"PR-AUC: {average_precision_score(y_test, y_proba):.3f}")

Best threshold: 0.75, F1: 0.451
PR-AUC: 0.402


XGBoost's best F1 (0.451 at threshold 0.75) already exceeds Random Forest's best result (0.418 at threshold 0.25) — consistent with gradient boosting's advantage over bagging on this kind of task, where each tree explicitly corrects the previous ones' errors rather than averaging independent trees.

### Hyperparameter Tuning

`RandomizedSearchCV` (30 iterations, 3-fold CV, optimizing PR-AUC) is used instead of a full grid search for tractability. PR-AUC is used as the tuning target rather than F1, since F1 depends on a decision threshold that is only selected *after* training — PR-AUC evaluates ranking quality across all thresholds at once.

In [63]:


param_dist = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [200, 300, 400, 500],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
        random_state=42,
        eval_metric='aucpr'
    ),
    param_distributions=param_dist,
    n_iter=30,
    scoring='average_precision',
    cv=3,
    random_state=42,
    n_jobs=-1
)
xgb_search.fit(X_train, y_train)
print(f"Best params: {xgb_search.best_params_}")
print(f"Best CV PR-AUC: {xgb_search.best_score_:.3f}")

Best params: {'subsample': 0.9, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best CV PR-AUC: 0.396


In [64]:
best_xgb = xgb_search.best_estimator_
y_proba_tuned = best_xgb.predict_proba(X_test)[:, 1]

print(f"Tuned PR-AUC on test: {average_precision_score(y_test, y_proba_tuned):.3f}")

best_f1_tuned = 0
best_threshold_tuned = 0.5
for threshold in np.arange(0.05, 0.95, 0.05):
    y_pred_t = (y_proba_tuned >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1_tuned:
        best_f1_tuned = f1
        best_threshold_tuned = threshold

print(f"Best threshold: {best_threshold_tuned:.2f}, F1: {best_f1_tuned:.3f}")

Tuned PR-AUC on test: 0.414
Best threshold: 0.80, F1: 0.461


Tuning improved PR-AUC from 0.402 to 0.414 and F1 from 0.451 to 0.461 — a real but modest gain. This is expected: it confirms the model was already close to what the data supports, rather than being poorly configured.

### LightGBM — Baseline

In [65]:
lgbm = LGBMClassifier(
    learning_rate=0.1,
    n_estimators=300,
    max_depth=5,
    scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)
print(classification_report(y_test, y_pred_lgbm))

              precision    recall  f1-score   support

           0       0.98      0.81      0.89     27876
           1       0.22      0.75      0.34      2002

    accuracy                           0.81     29878
   macro avg       0.60      0.78      0.61     29878
weighted avg       0.93      0.81      0.85     29878



In [66]:
y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]

best_f1_lgbm = 0
best_threshold_lgbm = 0.5

for threshold in np.arange(0.05, 0.95, 0.05):
    y_pred_t = (y_proba_lgbm >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1_lgbm:
        best_f1_lgbm = f1
        best_threshold_lgbm = threshold

print(f"Best threshold: {best_threshold_lgbm:.2f}, F1: {best_f1_lgbm:.3f}")
print(f"PR-AUC: {average_precision_score(y_test, y_proba_lgbm):.3f}")

Best threshold: 0.80, F1: 0.449
PR-AUC: 0.405


### LightGBM

Same tuning approach as XGBoost, with `n_jobs=1` set on the estimator to avoid CPU oversubscription against `RandomizedSearchCV`'s own parallelism (`n_jobs=-1`), which otherwise causes severe slowdowns.

In [67]:
param_dist_lgbm = {
    'num_leaves': [15, 31, 50, 70],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [200, 300, 400, 500],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [10, 20, 30]
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(
        scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
        random_state=42,
        verbose=-1,
        n_jobs=1
    ),
    param_distributions=param_dist_lgbm,
    n_iter=30,
    scoring='average_precision',
    cv=3,
    random_state=42,
    n_jobs=-1
)
lgbm_search.fit(X_train, y_train)
print(f"Best params: {lgbm_search.best_params_}")
print(f"Best CV PR-AUC: {lgbm_search.best_score_:.3f}")

Best params: {'subsample': 0.7, 'num_leaves': 70, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 7, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best CV PR-AUC: 0.395


In [68]:
best_lgbm = lgbm_search.best_estimator_
y_proba_lgbm_tuned = best_lgbm.predict_proba(X_test)[:, 1]

print(f"Tuned LightGBM PR-AUC on test: {average_precision_score(y_test, y_proba_lgbm_tuned):.3f}")

best_f1_lgbm_tuned = 0
best_threshold_lgbm_tuned = 0.5
for threshold in np.arange(0.05, 0.95, 0.05):
    y_pred_t = (y_proba_lgbm_tuned >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1_lgbm_tuned:
        best_f1_lgbm_tuned = f1
        best_threshold_lgbm_tuned = threshold

print(f"Best threshold: {best_threshold_lgbm_tuned:.2f}, F1: {best_f1_lgbm_tuned:.3f}")

Tuned LightGBM PR-AUC on test: 0.409
Best threshold: 0.75, F1: 0.456


### Random Forest (carried over from previous notebook)

Included here for direct comparison under identical train/test conditions. Full class-weighting and threshold exploration is documented in `01_eda_and_cleaning.ipynb`; only the final configuration is reproduced below.

In [69]:
random_forest_final = RandomForestClassifier(class_weight='balanced_subsample', random_state=42)
random_forest_final.fit(X_train, y_train)

y_proba_rf = random_forest_final.predict_proba(X_test)[:, 1]

threshold_rf = 0.25
y_pred_rf = (y_proba_rf >= threshold_rf).astype(int)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print(f"PR-AUC: {average_precision_score(y_test, y_proba_rf):.3f}")

[[26570  1306]
 [ 1129   873]]
              precision    recall  f1-score   support

           0       0.96      0.95      0.96     27876
           1       0.40      0.44      0.42      2002

    accuracy                           0.92     29878
   macro avg       0.68      0.69      0.69     29878
weighted avg       0.92      0.92      0.92     29878

PR-AUC: 0.356


## Summary

| Model | PR-AUC | F1 (best threshold) | Threshold |
|---|---|---|---|
| Random Forest | 0.356 | 0.418 | 0.25 |
| XGBoost (default) | 0.402 | 0.451 | 0.75 |
| LightGBM (tuned) | 0.409 | 0.456 | 0.75 |
| XGBoost (tuned) | 0.414 | 0.461 | 0.80 |

**Best model: XGBoost (tuned), F1 = 0.461.**

Random Forest lags noticeably behind both boosting methods, consistent with bagging's disadvantage against boosting on this kind of imbalanced, nonlinear problem. XGBoost and LightGBM, however, land within a hundredth of each other regardless of tuning — and hyperparameter tuning itself only added ~0.01 to each. This convergence across algorithms and configurations points to a real ceiling in this dataset (F1 ≈ 0.45), set by the information available in the features rather than by model choice.